In [0]:
%pip install atproto vaderSentiment

In [0]:
import time
from datetime import datetime
from atproto import Client
from pyspark.sql.types import StringType, StructField, StructType, TimestampType

# Login To Bluesky

In [0]:
client = Client()
profile=client.login("abdo25562.bsky.social", "2556250Ae@")

profile.handle

# Extract Posts from website

In [0]:
search_queries = [
    "gold",
]

date_ranges = [
    ("2026-01-01T00:00:00Z", "2026-01-31T23:59:59Z"),
    ("2026-02-01T00:00:00Z", "2026-02-28T23:59:59Z"),
    ("2026-03-01T00:00:00Z", "2026-03-31T23:59:59Z"),
    ("2026-04-01T00:00:00Z", "2026-04-30T23:59:59Z"),
    ("2026-05-01T00:00:00Z", "2026-05-31T23:59:59Z"),
    ("2026-06-01T00:00:00Z", "2026-06-30T23:59:59Z"),
    ("2026-07-01T00:00:00Z", "2026-07-31T23:59:59Z"),
    ("2026-08-01T00:00:00Z", "2026-08-31T23:59:59Z"),
    ("2026-09-01T00:00:00Z", "2026-09-30T23:59:59Z"),
    ("2026-10-01T00:00:00Z", "2026-10-31T23:59:59Z"),
    ("2026-11-01T00:00:00Z", "2026-11-16T23:59:59Z"),
]

schema = StructType([
    StructField("post_uri", StringType(), False),
    StructField("author_handle", StringType(), True),
    StructField("raw_text", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("search_topic", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), False),
])

spark.sql("CREATE DATABASE IF NOT EXISTS us_economy_db")
ingestion_time = datetime.utcnow()

MAX_PAGES_PER_MONTH = 50

for start_date, end_date in date_ranges:
    month_posts = []
    print(f"--- جاري سحب بيانات الفترة من {start_date[:10]} إلى {end_date[:10]} ---")

    for query in search_queries:
        cursor = None
        pages_fetched = 0

        while pages_fetched < MAX_PAGES_PER_MONTH:
            params = {
                "q": query,
                "limit": 100,
                "lang": "en",
                "since": start_date,
                "until": end_date,
            }
            if cursor:
                params["cursor"] = cursor

            try:
                response = client.app.bsky.feed.search_posts(params=params)
            except Exception as e:
                print(f"تنبيه أثناء سحب '{query}': {e} - الانتظار 5 ثوانٍ...")
                time.sleep(5)
                continue

            if not response.posts:
                break

            for post in response.posts:
                month_posts.append((
                    post.uri,
                    post.author.handle,
                    post.record.text,
                    post.record.created_at,
                    query,
                    ingestion_time,
                ))

            pages_fetched += 1
            cursor = response.cursor

            if not cursor:
                break

            time.sleep(0.5)

    if month_posts:
        df_month = spark.createDataFrame(month_posts, schema=schema)
        df_month.write.format("delta").mode("append").saveAsTable(
            "us_economy_db.bronze_bsky_posts"
        )
        print(f"تم حفظ {len(month_posts)} منشور للفترة {start_date[:7]} بنجاح.")

print("اكتمل سحب بيانات عام 2026 بالكامل!")


In [0]:
df_month.show()